# 5.3 Cell Cycle Modelling: c-MYC Shortens G1 (Figure 1E)

In the second half of this practical, we build and run the **37-species Cell Cycle ODE model** from *Norris et al. (Blood Cancer Journal, 2024)* to:

1. Recreate **Figure 1E (top)**: Simulate a complete steady-state division cycle for **WT (solid)** vs **MYC × 1.5 (dashed)** across **G1, S, G2, and M**!
2. Recreate **Figure 1E (bottom)**: Sweep **MYC overexpression from 1× to 100×** and plot the duration of **G1, S, G2, and M** separately—demonstrating visually that **G1 is the specific phase that gets shorter**!
3. **Rescue Exercise**: Screen candidate cell-cycle genes (such as **p27**, **Cyclin D**, **RB phosphorylation**, or **growth machinery `GM`**) to see which over- or under-expressions can **rescue c-MYC-driven G1 shortening** back toward WT—without arresting the cell cycle!

Click **⚡ Activate Live Editable Julia Cells** above to run and edit the simulations live!

## The Cell Cycle Pathway Recap

![Cell cycle phases and c-MYC regulation](../images/cell-cycle-pathway.svg)

Watch how each phase transition is identified quantitatively from the cycling protein curves:
- **G1 → S:** `Cdh1` drops below 50% of its peak (cells commit past the restriction point!)
- **S → G2:** `CycA` reaches 97.5% of its peak
- **G2 → M:** `CycB` hits its peak right before mitotic degradation
- **M (Division):** `Cdh1` rises past `0.2`, triggering a **division event** (`Mass → 0.5 × Mass`, `GM → 0.5 × GM`)!

## Step 1: Load Packages & Compile the 37-Species Cell Cycle Model

Again, we load **only** the four precompiled packages (`DifferentialEquations`, `Plots`, `CSV`, `DataFrames`) so startup is instant and memory usage stays well below 2 GB!

In [ ]:
if isdir("/home/jovyan/work/cellCycle")
    cd("/home/jovyan/work/cellCycle")
elseif isdir("cellCycle")
    cd("cellCycle")
end
@assert isfile("reactions_cellCycle.csv") "Cell cycle model files not found in $(pwd())."

using DifferentialEquations
using Plots
using CSV
using DataFrames

# Generate the 37-species Cell Cycle ODE model in "scan" mode
run(`python3 csv2model-multiscale.py reactions_cellCycle.csv parameters_cellCycle.csv rateLaws_cellCycle.csv odeCellCycle.jl scan`)

odeCCFun = include("odeCellCycle.jl")
include("variableNames.jl")
include("scanIncludes.jl")

println("Cell Cycle model ready: ", length(syms), " species, ", length(parameterList), " parameters.")

Cell Cycle model ready: 37 species, 82 parameters.

## Step 2: Set Up Division Callbacks, Phase-Timing Detector & Simulation Helper

This cell defines:
- The three `ContinuousCallback` biological events (RB switch `r31switch`, mitotic division when `Cdh1` rises past `0.2`, and low-mass arrest switch `newSwitch`)—automatically terminating once 5 consecutive cell divisions converge to an identical limit cycle!
- `measure_phases(...)` — calculates **G1**, **S**, **G2**, and **M** durations in minutes (and cleanly reports **cell-cycle arrest** if a strong rescue drug stops division altogether!).
- `run_cell_cycle(myc_mult; mods=Dict(...))` — runs any MYC multiplier + optional gene rescue without recompiling!

In [ ]:
const f_cc = ODEFunction(odeCCFun)
const maxTimeSS_cc = 100000.0

const idx_Mass   = findfirst(isequal("Mass"), syms)
const idx_GM     = findfirst(isequal("GM"), syms)
const idx_NewSw  = findfirst(isequal("newSwitch"), syms)
const idx_Rb     = findfirst(isequal("Rb"), syms)
const idx_E2FRb  = findfirst(isequal("E2F_Rb"), syms)
const idx_pE2FRb = findfirst(isequal("pE2F_Rb"), syms)
const idx_HypoRb = findfirst(isequal("HypoP_Rb"), syms)
const idx_r31Sw  = findfirst(isequal("r31switch"), syms)
const idx_Cdh1   = findfirst(isequal("Cdh1"), syms)
const idx_cycE   = findfirst(isequal("cycE"), syms)
const idx_cycA   = findfirst(isequal("cycA"), syms)
const idx_cycB   = findfirst(isequal("cycB"), syms)
const idx_cycD   = findfirst(isequal("cycD"), syms)

const idx_MT1 = findfirst(isequal("previousMass1"), syms)
const idx_MT2 = findfirst(isequal("previousMass2"), syms)
const idx_MT3 = findfirst(isequal("previousMass3"), syms)
const idx_MT4 = findfirst(isequal("previousMass4"), syms)
const idx_MT5 = findfirst(isequal("previousMass5"), syms)
const idx_MTT1 = findfirst(isequal("previousMassTime1"), syms)
const idx_MTT2 = findfirst(isequal("previousMassTime2"), syms)
const idx_MTT3 = findfirst(isequal("previousMassTime3"), syms)
const idx_MTT4 = findfirst(isequal("previousMassTime4"), syms)
const idx_MTT5 = findfirst(isequal("previousMassTime5"), syms)

# 1. RB restriction-point switch callback
function cond_rb(u, t, integrator)
    num = u[idx_Rb] + u[idx_E2FRb] + u[idx_pE2FRb]
    den = num + u[idx_HypoRb]
    return (num / max(den, 1e-12)) - 0.8
end
aff_rb_pos!(integrator) = (integrator.u[idx_r31Sw] = 0.0)
aff_rb_neg!(integrator) = (integrator.u[idx_r31Sw] = 1.0)

# 2. Mitotic division event when Cdh1 crosses 0.2 upwards
function cond_cdh1(u, t, integrator)
    return u[idx_Cdh1] - 0.2
end
function aff_cdh1!(integrator)
    integrator.u[idx_Mass] *= 0.5
    integrator.u[idx_GM]   *= 0.5
    integrator.u[idx_NewSw] = integrator.u[idx_Mass] < 0.5 ? 0.0 : 1.0

    # Record last 5 post-division masses & timestamps; stop once limit cycle converges
    integrator.u[idx_MT5]  = integrator.u[idx_MT4]
    integrator.u[idx_MT4]  = integrator.u[idx_MT3]
    integrator.u[idx_MT3]  = integrator.u[idx_MT2]
    integrator.u[idx_MT2]  = integrator.u[idx_MT1]
    integrator.u[idx_MT1]  = round(integrator.u[idx_Mass], digits=3)

    integrator.u[idx_MTT5] = integrator.u[idx_MTT4]
    integrator.u[idx_MTT4] = integrator.u[idx_MTT3]
    integrator.u[idx_MTT3] = integrator.u[idx_MTT2]
    integrator.u[idx_MTT2] = integrator.u[idx_MTT1]
    integrator.u[idx_MTT1] = integrator.t

    peaks = (integrator.u[idx_MT5], integrator.u[idx_MT4], integrator.u[idx_MT3],
             integrator.u[idx_MT2], integrator.u[idx_MT1])
    if peaks[1] > 0.0 && all(==(peaks[1]), peaks)
        terminate!(integrator)
    end
end

# 3. Minimum mass growth switch
cond_mass(u, t, integrator) = u[idx_Mass] - 0.5
aff_mass_pos!(integrator) = (integrator.u[idx_NewSw] = 1.0)
aff_mass_neg!(integrator) = (integrator.u[idx_NewSw] = 0.0)

const cc_cbs = CallbackSet(
    ContinuousCallback(cond_rb, aff_rb_pos!, aff_rb_neg!),
    ContinuousCallback(cond_cdh1, aff_cdh1!, nothing),
    ContinuousCallback(cond_mass, aff_mass_pos!, aff_mass_neg!),
)

function initConditionsCellCycle(syms, myc=40.0)
    y0 = zeros(length(syms))
    inits = Dict(
        "ERG" => 0.0121809, "p27_cycA_Cdk2" => 0.0356927, "p27" => 0.00922806,
        "Cdc20" => 0.00220177, "p27_cycE_Cdk2" => 0.000542587, "cycE" => 0.0229112,
        "cycA" => 1.4094, "cycB" => 2.72898, "p27_cycD_Cdk2" => 0.010976,
        "cycD" => 0.43929, "Cdh1" => 0.000653278, "DRG" => 0.900533,
        "PPX" => 1.0, "IEP" => 0.154655, "Cdc20t" => 2.36733,
        "E2F_Rb" => 0.00478911, "E2F" => 0.989986, "HypoP_Rb" => 9.97574,
        "pE2F_Rb" => 0.0192822, "pE2F" => 3.98594, "GM" => 1.35565,
        "r31switch" => 1.0, "cMyc" => Float64(myc), "Mass" => 1.0,
        "newSwitch" => 1.0, "Rb" => 0.0, "cMyct" => 40.0,
    )
    for (k, v) in inits
        idx = findfirst(isequal(k), syms)
        if !isnothing(idx)
            y0[idx] = v
        end
    end
    return y0
end

const cc_alg = if isdefined(Main, :AutoFiniteDiff)
    AutoTsit5(Rosenbrock23(autodiff=AutoFiniteDiff()))
elseif isdefined(DifferentialEquations, :AutoFiniteDiff)
    AutoTsit5(Rosenbrock23(autodiff=DifferentialEquations.AutoFiniteDiff()))
else
    AutoTsit5(Rosenbrock23())
end

function measure_phases(t_lc::Vector{Float64}, mat_lc::Matrix{Float64})
    cdh1 = mat_lc[idx_Cdh1, :]
    cyca = mat_lc[idx_cycA, :]
    cycb = mat_lc[idx_cycB, :]

    # G1/S: Cdh1 falls below 50% of its peak after reaching 95% peak
    max_cdh1 = maximum(cdh1)
    i_hi = findfirst(x -> x > 0.95 * max_cdh1, cdh1)
    isnothing(i_hi) && return (G1=NaN, S=NaN, G2=NaN, M=NaN, total=NaN, G1S=NaN, SG2=NaN, G2M=NaN, arrested=true)
    i_drop_rel = findfirst(x -> x < 0.5 * max_cdh1, @view cdh1[i_hi:end])
    isnothing(i_drop_rel) && return (G1=NaN, S=NaN, G2=NaN, M=NaN, total=NaN, G1S=NaN, SG2=NaN, G2M=NaN, arrested=true)
    i_g1s = min(i_hi + i_drop_rel - 1, length(t_lc))
    t_g1s = round(t_lc[i_g1s])

    # S/G2: CycA reaches 97.5% of its post-G1/S maximum
    max_cyca = maximum(@view cyca[i_g1s:end])
    i_sg2 = findfirst(x -> x > 0.975 * max_cyca, cyca)
    isnothing(i_sg2) && return (G1=NaN, S=NaN, G2=NaN, M=NaN, total=NaN, G1S=NaN, SG2=NaN, G2M=NaN, arrested=true)
    t_sg2 = round(t_lc[i_sg2])

    # G2/M: CycB hits its peak after S/G2
    max_cycb = maximum(@view cycb[i_sg2:end])
    i_g2m = findfirst(x -> x == max_cycb, cycb)
    isnothing(i_g2m) && return (G1=NaN, S=NaN, G2=NaN, M=NaN, total=NaN, G1S=NaN, SG2=NaN, G2M=NaN, arrested=true)
    t_g2m = round(t_lc[i_g2m])

    t_total = round(t_lc[end])
    return (
        G1 = t_g1s,
        S  = t_sg2 - t_g1s,
        G2 = t_g2m - t_sg2,
        M  = t_total - t_g2m,
        total = t_total,
        G1S = t_g1s, SG2 = t_sg2, G2M = t_g2m,
        arrested = false,
    )
end

function run_cell_cycle(myc_mult::Real; mods::Dict{String, <:Real}=Dict{String, Float64}())
    for k in keys(modify)
        modify[k] = 1.0
    end
    modify["k1_myc"] = Float64(myc_mult)
    for (k, v) in mods
        if !haskey(modify, k)
            error("Parameter '$(k)' not found! Check the Cell Cycle Target Genes table.")
        end
        modify[k] = Float64(v)
    end

    y0 = initConditionsCellCycle(syms, 40.0)
    prob = ODEProblem(f_cc, y0, (0.0, maxTimeSS_cc))
    sol  = solve(prob, cc_alg; callback=cc_cbs, abstol=1e-5, reltol=1e-3, saveat=1.0)

    sol_mat = Float64.(reduce(hcat, sol.u))
    t_prev2 = sol_mat[idx_MTT2, end]
    if t_prev2 <= 0.0
        # Cell arrested without completing steady-state divisions
        return (
            t_hours = sol.t ./ 60.0,
            mat_lc  = sol_mat,
            phases  = (G1=NaN, S=NaN, G2=NaN, M=NaN, total=NaN, G1S=NaN, SG2=NaN, G2M=NaN, arrested=true),
        )
    end

    start_idx = findfirst(x -> x > t_prev2, sol.t)
    t_lc   = sol.t[start_idx:end] .- sol.t[start_idx]
    mat_lc = sol_mat[:, start_idx:end]
    phases = measure_phases(t_lc, mat_lc)
    return (t_hours = t_lc ./ 60.0, mat_lc = mat_lc, phases = phases)
end

println("Cell cycle simulation & phase-duration helper ready!")

Cell cycle simulation & phase-duration helper ready!

## Step 3: Recreate Figure 1E (Top) — WT vs MYC × 1.5 Across G1, S, G2, M

Run the cell below to plot a single steady-state division cycle for **WT (solid lines)** vs **MYC × 1.5 (dashed lines)** overlaid on the shaded **G1, S, G2, and M** phase intervals!
Watch how the dashed **MYC × 1.5** curves shift left during **G1** (earlier `Cdh1` drop and earlier `CycE`/`CycA` rise!), making the whole division cycle finish earlier.

In [ ]:
cc_wt  = run_cell_cycle(1.0)
cc_myc = run_cell_cycle(1.5)

println("WT phase durations (mins):        G1=$(cc_wt.phases.G1), S=$(cc_wt.phases.S), G2=$(cc_wt.phases.G2), M=$(cc_wt.phases.M)  (Total = $(cc_wt.phases.total) min)")
println("MYC × 1.5 phase durations (mins): G1=$(cc_myc.phases.G1), S=$(cc_myc.phases.S), G2=$(cc_myc.phases.G2), M=$(cc_myc.phases.M)  (Total = $(cc_myc.phases.total) min)")

gr()
p_e_top = plot(
    xlabel="Time (h)", ylabel="Simulated Abundance (AU)",
    title="Figure 1E (Top): Cell Cycle — WT (solid) vs MYC × 1.5 (dashed)",
    xlims=(0, cc_wt.phases.total / 60.0 + 0.15),
    ylims=(0, 4.6),
    grid=false,
    size=(820, 400),
    legend=:topleft,
    xtickfontsize=11, ytickfontsize=11, xguidefontsize=13, yguidefontsize=13, legendfontsize=9,
)

# Shaded WT G1, S, G2, M phase backgrounds
g1_end = cc_wt.phases.G1S / 60.0
s_end  = cc_wt.phases.SG2 / 60.0
g2_end = cc_wt.phases.G2M / 60.0
m_end  = cc_wt.phases.total / 60.0
vspan!(p_e_top, [0.0, g1_end],    color=:purple, alpha=0.10, label=false)
vspan!(p_e_top, [g1_end, s_end],  color=:green,  alpha=0.10, label=false)
vspan!(p_e_top, [s_end, g2_end],  color=:orange, alpha=0.12, label=false)
vspan!(p_e_top, [g2_end, m_end],  color=:blue,   alpha=0.10, label=false)

annotate!(p_e_top, g1_end / 2,            4.35, text("G1", 12, :purple, :bold))
annotate!(p_e_top, (g1_end + s_end) / 2,  4.35, text("S",  12, :forestgreen, :bold))
annotate!(p_e_top, (s_end + g2_end) / 2,  4.35, text("G2", 12, :darkorange, :bold))
annotate!(p_e_top, (g2_end + m_end) / 2,  4.35, text("M",  12, :royalblue, :bold))

species_specs = [
    ("Cdh1", idx_Cdh1, :forestgreen),
    ("CycA", idx_cycA, :purple),
    ("Mass", idx_Mass, :goldenrod),
    ("CycB", idx_cycB, :red),
    ("CycE", idx_cycE, :olive),
    ("CycD", idx_cycD, :royalblue),
]

for (lab, idx, col) in species_specs
    plot!(p_e_top, cc_wt.t_hours,  cc_wt.mat_lc[idx, :],  color=col, lw=2.5, linestyle=:solid, label=lab)
    plot!(p_e_top, cc_myc.t_hours, cc_myc.mat_lc[idx, :], color=col, lw=2.0, linestyle=:dash,  alpha=0.75, label=false)
end

display(p_e_top)

## Step 4: Recreate Figure 1E (Bottom) — Which Phase Gets Shorter as MYC Increases?

Now sweep **MYC Multiplier** across `[1, 1.5, 2, 5, 10, 50, 100]` and plot the duration (in minutes) of **G1**, **S**, **G2**, and **M** side-by-side!
Look at the four panels: **G1 drops steeply as c-MYC rises, whereas S, G2, and M stay virtually flat!**

In [ ]:
myc_mults = [1.0, 1.5, 2.0, 5.0, 10.0, 50.0, 100.0]
phase_df = DataFrame(MYC=Float64[], G1=Float64[], S=Float64[], G2=Float64[], M=Float64[], Total=Float64[])

for m in myc_mults
    res = run_cell_cycle(m)
    ph = res.phases
    push!(phase_df, (m, ph.G1, ph.S, ph.G2, ph.M, ph.total))
end

display(phase_df)

xt = ["1", "1.5", "2", "5", "10", "50", "100"]
xs = 1:length(myc_mults)

p_g1 = plot(xs, phase_df.G1, lw=3, marker=:circle, ms=5, color=:purple,
            xticks=(xs, xt), xrotation=60, ylims=(280, 315), title="G1", ylabel="Time (mins)", legend=false, grid=false)
p_s  = plot(xs, phase_df.S,  lw=3, marker=:circle, ms=5, color=:forestgreen,
            xticks=(xs, xt), xrotation=60, ylims=(270, 305), title="S",  legend=false, grid=false)
p_g2 = plot(xs, phase_df.G2, lw=3, marker=:circle, ms=5, color=:darkorange,
            xticks=(xs, xt), xrotation=60, ylims=(80, 115),  title="G2", legend=false, grid=false)
p_m  = plot(xs, phase_df.M,  lw=3, marker=:circle, ms=5, color=:royalblue,
            xticks=(xs, xt), xrotation=60, ylims=(25, 60),   title="M",  legend=false, grid=false)

p_e_bot = plot(p_g1, p_s, p_g2, p_m, layout=(1, 4), size=(900, 290),
               plot_title="Figure 1E (Bottom): Phase Durations vs MYC Multiplier",
               xlabel="MYC")
display(p_e_bot)

## Cell Cycle Rescue Targets Reference Table

Can we **rescue** c-MYC-driven G1 shortening—bringing **G1 duration** back up to WT levels (~303–309 mins)—by **over-expressing** a tumour suppressor (`> 1.0`) or **inhibiting** a cyclin/kinase/growth target (`< 1.0`)?

| Gene / Protein | Role in Cell Cycle | Parameter Name in Julia | To Slow G1 (Rescue c-MYC OE) |
| :--- | :--- | :--- | :--- |
| **p27 (`CDKN1B`)** | CDK inhibitor tumour suppressor (G1/S brake) | `"k2_5"` | **Over-express** (e.g. `1.3` – `2.0`) |
| **Cyclin D (`cycD`)** | G1 driver activating CDK4/6 *(palbociclib pathway)* | `"k2_9"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
| **RB phosphorylation** | CDK4/6/2 phosphorylation releasing E2F from RB1 | `"k1_20"` | **Inhibit** (e.g. `0.5` – `0.8`) |
| **RB1–E2F binding** | Tumour suppressor holding E2F inactive in G1 | `"k1_26"` | **Over-express / boost** (e.g. `1.5` – `3.0`) |
| **Growth machinery (`GM`)** | Ribosome biogenesis directly driven by c-MYC | `"k1_GMprod"` | **Under-express / inhibit** (e.g. `0.7` – `0.9`) |
| **Cyclin E (`cycE`)** | G1/S transition cyclin | `"k3_7"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
| **Cyclin A (`cycA`)** | S-phase progression cyclin | `"k2_29"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
| **Cyclin B (`cycB`)** | G2/M mitotic cyclin | `"k2_1a"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |

## Exercise: Rescue c-MYC Overexpression (`MYC × 2.0`)

In a `MYC × 2.0` lymphoma cell, G1 shortens significantly, driving faster proliferation.

**Your task:** Edit `rescue_mods` in the starter cell below to test whether over- or under-expressing a candidate target gene can **restore WT-like G1 duration**—and watch what happens if you push a G1 brake *too* hard (can you trigger complete **cell-cycle arrest**?!).

````{admonition} 📋 Quick Reference: Cell Cycle Target Genes
:class: note, dropdown

| Gene / Protein | Role in Cell Cycle | Parameter Name in Julia | To Slow G1 (Rescue c-MYC OE) |
| :--- | :--- | :--- | :--- |
| **p27 (`CDKN1B`)** | CDK inhibitor tumour suppressor (G1/S brake) | `"k2_5"` | **Over-express** (e.g. `1.3` – `2.0`) |
| **Cyclin D (`cycD`)** | G1 driver activating CDK4/6 *(palbociclib pathway)* | `"k2_9"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
| **RB phosphorylation** | CDK4/6/2 phosphorylation releasing E2F from RB1 | `"k1_20"` | **Inhibit** (e.g. `0.5` – `0.8`) |
| **RB1–E2F binding** | Tumour suppressor holding E2F inactive in G1 | `"k1_26"` | **Over-express / boost** (e.g. `1.5` – `3.0`) |
| **Growth machinery (`GM`)** | Ribosome biogenesis directly driven by c-MYC | `"k1_GMprod"` | **Under-express / inhibit** (e.g. `0.7` – `0.9`) |
| **Cyclin E (`cycE`)** | G1/S transition cyclin | `"k3_7"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
| **Cyclin A (`cycA`)** | S-phase progression cyclin | `"k2_29"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
| **Cyclin B (`cycB`)** | G2/M mitotic cyclin | `"k2_1a"` | **Under-express / inhibit** (e.g. `0.5` – `0.8`) |
````

In [ ]:
# Exercise starter code — edit rescue_mods below to rescue MYC × 2.0 G1 shortening!
test_myc    = 2.0
rescue_name = "MYC × 2.0 + My Rescue"
rescue_mods = Dict{String, Float64}()   # <-- Try e.g. Dict("k2_5" => 1.5) or Dict("k2_9" => 0.6)!

res_wt     = run_cell_cycle(1.0)
res_mut    = run_cell_cycle(test_myc)
res_rescue = run_cell_cycle(test_myc; mods=rescue_mods)

rows = [
    ("WT",                       res_wt.phases),
    ("MYC × $(test_myc) (untreated)", res_mut.phases),
    (rescue_name,                res_rescue.phases),
]

display(DataFrame(
    Condition = [r[1] for r in rows],
    G1_mins   = [r[2].G1 for r in rows],
    S_mins    = [r[2].S  for r in rows],
    G2_mins   = [r[2].G2 for r in rows],
    M_mins    = [r[2].M  for r in rows],
    Total_min = [r[2].total for r in rows],
    Status    = [r[2].arrested ? "⚠️ Arrested (no division)" : "Cycling" for r in rows],
))

if !res_rescue.phases.arrested
    p_bar = bar(
        ["WT", "MYC × $(test_myc)", rescue_name],
        [res_wt.phases.G1, res_mut.phases.G1, res_rescue.phases.G1];
        fillcolor=[:steelblue, :orangered, :forestgreen],
        legend=false,
        ylabel="G1 Duration (mins)",
        title="G1 Duration Rescue Comparison",
        ylims=(270, max(320, res_rescue.phases.G1 + 10)),
        size=(680, 340),
        xtickfontsize=11, ytickfontsize=11, yguidefontsize=13,
    )
    hline!(p_bar, [res_wt.phases.G1], linestyle=:dash, color=:black, lw=1.5)
    display(p_bar)
else
    println("⚠️ Your perturbation completely arrested the cell cycle! Try a gentler multiplier (closer to 1.0) to restore WT G1 speed without total arrest.")
end

````{admonition} 💡 Hint — Rescuing c-MYC Overexpression
:class: tip, dropdown

Look at what controls **G1 length** specifically:
1. **Inhibiting Growth Machinery (`Dict("k1_GMprod" => 0.85)`):** Since c-MYC accelerates `GM` (ribosome/growth biosynthesis), dialling down `k1_GMprod` directly counteracts MYC!
2. **Boosting p27 (`Dict("k2_5" => 1.5)`):** Over-expressing the CDK inhibitor `p27` (`CDKN1B`) delays CDK2/4 activation and lengthens G1!
3. **Inhibiting Cyclin D (`Dict("k2_9" => 0.6)`) or RB phosphorylation (`Dict("k1_20" => 0.6)`):** Mimics clinical **CDK4/6 inhibitors** (like palbociclib) by holding RB1 active longer in G1!
4. **Compare with S/G2 targets (`k2_29` CycA or `k2_1a` CycB):** Do S/G2 targets rescue G1, or do they change a *different* phase of the cell cycle?
````

````{admonition} ✅ Reveal Solution — Rescuing c-MYC Overexpression
:class: note, dropdown

Targets acting directly in **G1**—boosting **p27** (`"k2_5"`), inhibiting **Cyclin D** (`"k2_9"`), inhibiting **RB phosphorylation** (`"k1_20"`), or reducing **growth machinery `GM`** (`"k1_GMprod"`)—specifically lengthen **G1** back toward WT (~303–309 mins), while stronger inhibition can arrest cycling cells in G1 altogether!

```julia
test_myc = 2.0
candidates = [
    ("WT",                       1.0,      Dict{String, Float64}()),
    ("MYC × 2 (untreated)",      test_myc, Dict{String, Float64}()),
    ("+ GM inh (0.85×)",         test_myc, Dict("k1_GMprod" => 0.85)),
    ("+ p27 OE (1.5×)",          test_myc, Dict("k2_5" => 1.5)),
    ("+ CycD inh (0.6×)",        test_myc, Dict("k2_9" => 0.6)),
    ("+ RB-phos / CDK4/6 (0.6×)",test_myc, Dict("k1_20" => 0.6)),
]

out_df = DataFrame(Condition=String[], G1=Float64[], S=Float64[], G2=Float64[], M=Float64[], Total=Float64[], Status=String[])
for (name, m, d) in candidates
    ph = run_cell_cycle(m; mods=d).phases
    push!(out_df, (name, ph.G1, ph.S, ph.G2, ph.M, ph.total, ph.arrested ? "Arrested" : "Cycling"))
end
display(out_df)

cycling = filter(r -> r.Status == "Cycling", out_df)
p_sol = bar(
    cycling.Condition, cycling.G1;
    xrotation=25, legend=false, fillcolor=:purple, alpha=0.8,
    ylabel="G1 Duration (mins)", title="Comparing Candidate Rescues for MYC × 2.0",
    ylims=(270, max(325, maximum(cycling.G1) + 10)), size=(780, 380),
)
hline!(p_sol, [out_df.G1[1]], linestyle=:dash, color=:black, lw=1.8)
display(p_sol)
```
````

---

## Wrapping Up & SSC Assessment Ideas

You have now modelled **both halves** of Norris et al. (*Blood Cancer Journal*, 2024):
- **Apoptosis (BCL2)** — anti-apoptotic (**AA**) resistance to programmed cell death (**Figure 1B**)
- **Cell Cycle (c-MYC)** — pro-proliferative (**PP**) shortening of the **G1** phase (**Figure 1E**)

Great ideas for your **6-minute SSC assessment presentation**:
1. **Why do Double-Hit (BCL2 + MYC) patients do so poorly?** Contrast what each oncogene does independently vs together, and discuss how patient-specific models predict AAPP prognosis!
2. **Rescue Dose-Response:** Sweep your favourite rescue target (e.g. `BAX` or `BCL2` inhibition in Apoptosis, or `p27` / `CycD` / `CDK4/6` inhibition in Cell Cycle) to find the exact rescue threshold—and in the cell cycle, the threshold where **rescue turns into G1 arrest**!
3. **Combination Therapy & Limitations:** What happens when a lymphoma cell has *multiple* co-occurring mutations (e.g. `BCL2` overexpression + `BAX` loss, or `MYC` overexpression + `RB1` loss)?